# Task 3: Deep-Dive Analysis & Interactive Dashboarding

## Objective
To perform a deep-dive analysis on customer and sales data, define key business KPIs, identify customer segments, and build an interactive Power BI dashboard for business decision-making.

## Dataset
Cleaned Sales Dataset

## Analysis Focus
Customer Segmentation and Sales Performance Analysis


In [7]:
import pandas as pd
import numpy as np

# 1. Load cleaned dataset
df = pd.read_csv('cleaned_dataset.csv')
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# Reference date for recency calculation (day after the latest order)
ref_date = df['Order_Date'].max() + pd.Timedelta(days=1)

# 2. RFM Calculation per Customer
rfm = df.groupby('Customer_ID').agg({
    'Order_Date': lambda x: (ref_date - x.max()).days,  # Recency
    'Order_ID': 'count',                                # Frequency
    'Total_Sales': 'sum'                                # Monetary
}).reset_index()

rfm.columns = ['Customer_ID', 'Recency_Days', 'Frequency', 'Monetary_Value']

# 3. RFM Scoring (1 to 4 quantiles)
rfm['R_Score'] = pd.qcut(rfm['Recency_Days'], q=4, labels=[4, 3, 2, 1]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary_Value'], q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['RFM_Sum'] = rfm['R_Score'] + rfm['M_Score']

# 4. Assign Segments based on RFM Score
def assign_segment(score):
    if score >= 7:
        return 'Champions / VIP'
    elif score >= 5:
        return 'Loyal Spenders'
    elif score >= 3:
        return 'Potential Regulars'
    else:
        return 'At Risk / Churn Risk'

rfm['Customer_Segment'] = rfm['RFM_Sum'].apply(assign_segment)

# 5. Merge back with transaction data for Power BI
enriched_df = df.merge(rfm[['Customer_ID', 'Recency_Days', 'Customer_Segment']], on='Customer_ID', how='left')

# 6. Save the enriched dataset
enriched_df.to_csv('task3_enriched_dataset.csv', index=False)
print("Enriched dataset successfully created: task3_enriched_dataset.csv")

Enriched dataset successfully created: task3_enriched_dataset.csv
